# 예제 01. 순서가 있는 데이터 만들기
빅데이터프로그래밍 · 11주차

## 목표
- 시계열 데이터를 만들고 그린다
- 순서를 섞으면 무엇이 사라지는지 확인한다
- 사인파 · 기온 · 전력 사용량 형태를 만들어 본다

이미지는 위치가 중요했습니다. 시계열은 **순서**가 중요합니다.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)


## 1. 사인파 — 가장 단순한 시계열


In [ ]:
t = np.arange(0, 200)
sine = np.sin(t * 0.1)

plt.figure(figsize=(11, 3))
plt.plot(t, sine)
plt.title("sine wave"); plt.xlabel("time"); plt.grid(alpha=.3)
plt.show()

print("길이:", len(sine))
print("앞 10개:", sine[:10].round(3))


## 2. 순서를 섞으면 — 규칙이 사라집니다


In [ ]:
shuffled = sine.copy()
np.random.shuffle(shuffled)

fig, ax = plt.subplots(1, 2, figsize=(13, 3))
ax[0].plot(sine); ax[0].set_title("원본 — 규칙이 보입니다"); ax[0].grid(alpha=.3)
ax[1].plot(shuffled); ax[1].set_title("순서를 섞음 — 규칙이 사라졌습니다"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print("평균과 표준편차는 그대로:")
print(f"  원본 {sine.mean():.4f} / {sine.std():.4f}")
print(f"  섞음 {shuffled.mean():.4f} / {shuffled.std():.4f}")


값의 집합은 같은데 의미가 완전히 다릅니다. **순서 자체가 정보**입니다.
9주차에서 이미지를 좌우로 뒤집어도 옷은 옷이었던 것과 다릅니다.


## 3. 여러 형태의 시계열


In [ ]:
t = np.arange(0, 365)

series = {
    "사인파":      np.sin(t * 0.1),
    "기온 (계절)":  15 + 12 * np.sin((t - 100) * 2*np.pi/365) + np.random.randn(365)*1.5,
    "전력 (주기+추세)": 100 + t*0.05 + 15*np.sin(t * 2*np.pi/7) + np.random.randn(365)*3,
    "센서 (잡음 많음)": np.cumsum(np.random.randn(365)*0.5),
}

fig, axes = plt.subplots(2, 2, figsize=(13, 6))
for ax, (name, s) in zip(axes.flatten(), series.items()):
    ax.plot(s, linewidth=1)
    ax.set_title(name); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()


| 구성 요소 | 예 |
| --- | --- |
| 추세 | 전력 사용량이 해마다 늘어남 |
| 주기 | 기온의 계절 변화, 전력의 주간 패턴 |
| 잡음 | 측정 오차 |


## 4. 자기상관 — 이전 값이 이후 값에 영향을 줍니다
통계학의 자기상관 개념 그대로입니다.


In [ ]:
import pandas as pd

s = pd.Series(series["기온 (계절)"])
rows = []
for lag in [1, 3, 7, 30, 90, 180]:
    rows.append({"lag": lag, "자기상관": round(s.autocorr(lag), 4)})
pd.DataFrame(rows)


In [ ]:
# 섞은 데이터는 자기상관이 사라집니다
sh = pd.Series(np.random.permutation(series["기온 (계절)"]))
print("원본 lag=1 자기상관:", round(s.autocorr(1), 4))
print("섞음 lag=1 자기상관:", round(sh.autocorr(1), 4))


## 5. 학습·검증 분할은 시간 순서로
시계열은 무작위로 섞어 나누면 안 됩니다. 미래 데이터로 과거를 예측하는 셈이 됩니다.


In [ ]:
data = series["사인파"]
n_train = int(len(data) * 0.8)
train, val = data[:n_train], data[n_train:]

plt.figure(figsize=(11, 3))
plt.plot(range(n_train), train, label="학습 (앞 80%)")
plt.plot(range(n_train, len(data)), val, label="검증 (뒤 20%)")
plt.legend(); plt.grid(alpha=.3); plt.title("시간 순서로 분할")
plt.show()

print(f"학습 {len(train)}개 · 검증 {len(val)}개")


## 직접 해보기
1. 주기가 다른 사인파를 두 개 더해 그려 보세요.
2. 추세와 주기와 잡음을 모두 가진 시계열을 만들어 보세요.
3. lag를 1부터 30까지 바꾸며 자기상관을 그래프로 그리세요.


In [ ]:
# 여기에 작성하세요
